In [16]:
import os
import numpy as np

# ✅ Path to your saved .npy file
npy_file_path = "../MediaPipe_landmarks/squat_back_res_bob_landmarks.npy"

# ✅ Load the NumPy array
landmarks_data = np.load(npy_file_path)

# ✅ Check its shape
print("Shape:", landmarks_data.shape)

# ✅ Inspect first frame
print("First frame landmarks:\n", landmarks_data)


Shape: (416, 33, 3)
First frame landmarks:
 [[[ 0.46329933  0.49737012  0.3187789 ]
  [ 0.45618325  0.50685656  0.29767832]
  [ 0.45603752  0.51235968  0.29753593]
  ...
  [ 0.89513958  0.37086928 -0.05050464]
  [ 0.85822541  0.59667176 -0.01041919]
  [ 0.85643995  0.36843732 -0.06706373]]

 [[ 0.46320304  0.49729779  0.30494311]
  [ 0.45603865  0.50673199  0.28187424]
  [ 0.45593578  0.51222658  0.28174549]
  ...
  [ 0.89837641  0.37115663 -0.04395538]
  [ 0.85896057  0.59893829  0.01516149]
  [ 0.85463279  0.36661473 -0.0559625 ]]

 [[ 0.46320304  0.49488536  0.29988247]
  [ 0.45606032  0.50435334  0.27670211]
  [ 0.45598093  0.50998962  0.27657324]
  ...
  [ 0.89973563  0.3715148  -0.04327255]
  [ 0.85916591  0.60004216  0.01740788]
  [ 0.8536185   0.36588523 -0.05533395]]

 ...

 [[ 0.4981654   0.51638877  0.37316555]
  [ 0.49354026  0.52322948  0.35504001]
  [ 0.49342546  0.52750349  0.35486835]
  ...
  [ 0.84263098  0.48104456  0.15237771]
  [ 0.88280225  0.6260373  -0.26034486]


In [ ]:
import sys, os

# Go two levels up to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)

print("Project root added to path:", project_root)

from Utils.utils.utils import *



# MediaPipe joint indices
HIP_L = 23
KNEE_L = 25
ANKLE_L = 27
TOE_L = 31
HEEL_L = 29

HIP_R = 24
KNEE_R = 26
ANKLE_R = 28
TOE_R = 32
HEEL_R = 30

SHOULDER_L = 11
ELBOW_L = 13
WRIST_L = 15
THUMB_L = 21
INDEXFINGER_L = 19
PINKY_L = 17

SHOULDER_R = 12
ELBOW_R = 14
WRIST_R = 16
THUMB_R = 22
INDEXFINGER_R = 20
PINKY_R = 18

NOSE = 0



def normalize_skeleton_with_virtual_joints(coords, lhip_idx, rhip_idx,
                                           lsho_idx, rsho_idx, eps=1e-8):
    """
    coords: (T, J, 3) raw 3D landmarks
    lhip_idx, rhip_idx: left/right hip indices
    lsho_idx, rsho_idx: left/right shoulder indices

    Returns:
      coords_norm: (T, J+2, 3) normalized coords including virtual pelvis and neck
      pelvis:      (T, 3) pelvis positions before centering
      neck:        (T, 3) neck positions before centering
    """

    T, J, _ = coords.shape

    # 1) Virtual mid-pelvis and mid-shoulder (neck proxy)
    left_hip  = coords[:, lhip_idx, :]    # (T, 3)
    right_hip = coords[:, rhip_idx, :]    # (T, 3)
    pelvis = (left_hip + right_hip) / 2.0 # (T, 3)

    left_sho  = coords[:, lsho_idx, :]    # (T, 3)
    right_sho = coords[:, rsho_idx, :]    # (T, 3)
    neck = (left_sho + right_sho) / 2.0   # (T, 3)

    # 2) Center all original joints on pelvis
    coords_centered = coords - pelvis[:, None, :]  # (T, J, 3)

    # 3) Also center virtual joints
    pelvis_centered = pelvis - pelvis              # becomes (T, 3) at origin
    neck_centered   = neck - pelvis                # neck relative to pelvis

    # 4) Stack virtual joints at the end: [J joints, pelvis, neck]
    pelvis_centered = pelvis_centered[:, None, :]  # (T, 1, 3)
    neck_centered   = neck_centered[:, None, :]    # (T, 1, 3)
    coords_with_virtual = np.concatenate(
        [coords_centered, pelvis_centered, neck_centered], axis=1
    )  # (T, J+2, 3)

    # 5) Compute scale as pelvis->neck distance
    # neck is last joint index: J+1
    neck_rel = coords_with_virtual[:, -1, :]                # (T, 3)
    scale = np.linalg.norm(neck_rel, axis=-1, keepdims=True) + eps  # (T, 1)

    # 6) Scale all joints
    coords_norm = coords_with_virtual / scale[:, None, :]   # (T, J+2, 3)

    return coords_norm, pelvis, neck



coords_norm, pelvis_raw, neck_raw = normalize_skeleton_with_virtual_joints(
    landmarks_data, HIP_L, HIP_R, SHOULDER_L, SHOULDER_R
)

Project root added to path: c:\Users\chris\OneDrive\Desktop\Fritidsprojekt\TempAISpotter\AI\OlympicAi


array([[[-0.92817521,  0.0306697 ,  1.60434967],
        [-0.96398385,  0.07840615,  1.49816998],
        [-0.96471719,  0.10609825,  1.49745344],
        ...,
        [ 1.05013747, -0.61812971, -0.33723901],
        [ 0.        ,  0.        ,  0.        ],
        [-0.69736924,  0.0444092 ,  0.71533479]],

       [[-1.00037098,  0.02930997,  1.64228498],
        [-1.03894997,  0.08011146,  1.51806309],
        [-1.03950395,  0.1096989 ,  1.51736982],
        ...,
        [ 1.10741024, -0.67439563, -0.30112908],
        [ 0.        ,  0.        ,  0.        ],
        [-0.75316002,  0.04491227,  0.65630234]],

       [[-1.01826558,  0.01479148,  1.64156742],
        [-1.05736056,  0.06661357,  1.514692  ],
        [-1.05779511,  0.09746316,  1.51398667],
        ...,
        [ 1.11863448, -0.69127785, -0.3026742 ],
        [ 0.        ,  0.        ,  0.        ],
        [-0.76759772,  0.04396852,  0.63942193]],

       ...,

       [[-0.65517172, -0.02657023,  1.64789634],
        [-0

In [ ]:
def compute_angle_features(landmarks):
    frames, joints, dims = landmarks.shape
    features = []

    for f in range(frames):
        lm = landmarks[f]

        # Example angles
        left_ankle = calculate_angle(lm[KNEE_L], lm[ANKLE_L], lm[TOE_L])
        right_ankle = calculate_angle(lm[KNEE_R], lm[ANKLE_R], lm[TOE_R])
        
        left_knee = calculate_angle(lm[HIP_L], lm[KNEE_L], lm[ANKLE_L])
        right_knee = calculate_angle(lm[HIP_R], lm[KNEE_R], lm[ANKLE_R])
        
        left_hip = calculate_angle(lm[SHOULDER_L], lm[HIP_L], lm[KNEE_L])
        right_hip = calculate_angle(lm[SHOULDER_R], lm[HIP_R], lm[KNEE_R])
        
        left_shoulder = calculate_angle(lm[ELBOW_L], lm[SHOULDER_L], lm[HIP_L])
        right_shoulder = calculate_angle(lm[ELBOW_R], lm[SHOULDER_R], lm[HIP_R])

        left_elbow = calculate_angle(lm[SHOULDER_L], lm[ELBOW_L], lm[WRIST_L])
        right_elbow = calculate_angle(lm[SHOULDER_R], lm[ELBOW_R], lm[WRIST_R])
        
        left_wrist = calculate_angle(lm[ELBOW_L], lm[WRIST_L], lm[PINKY_L])
        right_wrist = calculate_angle(lm[ELBOW_R], lm[WRIST_R], lm[PINKY_R])

        # Add more angles if you want a richer embedding

        features.append([
            left_ankle,
            right_ankle,
            left_knee,
            right_knee,
            left_hip,
            right_hip,
            left_shoulder,
            right_shoulder,
            left_wrist,
            right_wrist,
            right_elbow,
            left_elbow,
        ])

    return np.array(features)

new_arr = compute_angle_features(landmarks_data)
new_arr.shape

In [3]:
new_arr

array([[ 50.69289775,  40.43983666, 179.91583543, ..., 150.81647918,
         79.47146669,  67.6691427 ],
       [ 54.02938182,  46.32351744, 179.94149166, ..., 152.35136275,
         80.02924657,  68.08882426],
       [ 55.44479796,  49.26031584, 179.95721169, ..., 153.93392157,
         80.45086228,  68.0226951 ],
       ...,
       [ 80.41962056,  71.83612157, 174.32071517, ..., 152.77206141,
         64.12752976,  71.60402784],
       [ 80.40760187,  72.00715701, 174.2580263 , ..., 153.75303682,
         65.21503881,  72.39190523],
       [ 80.0090074 ,  70.8432439 , 174.05664392, ..., 154.83997744,
         66.17763667,  73.30844077]])

In [4]:
norm_arr = (new_arr - new_arr.mean(axis=0)) / new_arr.std(axis=0)
norm_arr

array([[-0.91411195, -0.60352639,  0.65129104, ...,  0.25368948,
         1.28258049,  0.57008646],
       [-0.52095773, -0.21832246,  0.65191618, ...,  0.50909567,
         1.37950807,  0.62718259],
       [-0.35417233, -0.0260506 ,  0.65229922, ...,  0.77243506,
         1.45277387,  0.61818596],
       ...,
       [ 2.58873307,  1.45198491,  0.51496034, ...,  0.57910035,
        -1.3837951 ,  1.10541305],
       [ 2.58731685,  1.46318258,  0.51343286, ...,  0.74233566,
        -1.19481442,  1.21260085],
       [ 2.54034851,  1.38698132,  0.50852598, ...,  0.92320368,
        -1.02754005,  1.33729213]])

In [5]:
velocity = np.diff(norm_arr, axis=0)
velocity

array([[ 3.93154223e-01,  3.85203931e-01,  6.25139505e-04, ...,
         2.55406188e-01,  9.69275790e-02,  5.70961253e-02],
       [ 1.66785401e-01,  1.92271867e-01,  3.83034459e-04, ...,
         2.63339397e-01,  7.32658008e-02, -8.99662758e-03],
       [ 1.49034839e-01,  1.42919027e-01,  7.06932227e-04, ...,
         1.71077433e-01,  4.40284266e-02, -3.22310769e-02],
       ...,
       [-1.20830199e-02,  3.14897179e-02,  2.31868242e-05, ...,
        -3.08846280e-02,  8.61240018e-02, -9.55377501e-02],
       [-1.41622049e-03,  1.11976715e-02, -1.52747703e-03, ...,
         1.63235306e-01,  1.88980677e-01,  1.07187807e-01],
       [-4.69683346e-02, -7.62012625e-02, -4.90688296e-03, ...,
         1.80868025e-01,  1.67274374e-01,  1.24691274e-01]])

In [6]:
left_knee  = norm_arr[:, 3]
right_knee = norm_arr[:, 4]
left_hip   = norm_arr[:, 5]
right_hip  = norm_arr[:, 6]



knee_symmetry = left_knee - right_knee
hip_symmetry  = left_hip - right_hip
knee_symmetry.shape, hip_symmetry.shape

((416,), (416,))

In [7]:
from scipy.signal import savgol_filter
angles_smooth = savgol_filter(norm_arr, window_length=11, polyorder=3, axis=0)
angles_smooth

array([[-0.88382271, -0.5615659 ,  0.6507668 , ...,  0.24251629,
         1.27267381,  0.58266541],
       [-0.56593408, -0.26498182,  0.65247408, ...,  0.51795002,
         1.38767922,  0.61079926],
       [-0.35184387, -0.05237425,  0.65304853, ...,  0.76312717,
         1.46306958,  0.60941943],
       ...,
       [ 2.6128552 ,  1.4280116 ,  0.5134002 , ...,  0.60812214,
        -1.40217345,  1.13390073],
       [ 2.59287045,  1.43170547,  0.51083503, ...,  0.68364756,
        -1.24575533,  1.18218938],
       [ 2.53006507,  1.4158443 ,  0.51008936, ...,  0.93852985,
        -0.99150971,  1.35461442]])

In [8]:
min_frames = velocity.shape[0]  # 415

# Remove last frames to make vectors match in dimensions
angles_smooth_trimmed = angles_smooth[:min_frames]
knee_symmetry_trimmed = knee_symmetry[:min_frames]
hip_symmetry_trimmed  = hip_symmetry[:min_frames]

# Add dimension for concatenation
knee_symmetry_trimmed = knee_symmetry_trimmed.reshape(-1, 1)
hip_symmetry_trimmed  = hip_symmetry_trimmed.reshape(-1, 1)


velocity.shape, angles_smooth_trimmed.shape, knee_symmetry_trimmed.shape, hip_symmetry_trimmed.shape

embedding = np.concatenate([
    angles_smooth_trimmed,
    velocity,                  # already 415
    knee_symmetry_trimmed,
    hip_symmetry_trimmed
], axis=1)

embedding.shape

(415, 26)

In [9]:
np.save("../embedding/squat_back_res_bob_angle_embedding.npy", embedding)

In [10]:
bob = np.load("../embedding/squat_back_res_bob_angle_embedding.npy")
bobby = np.load("../embedding/squat_back_res_bobby_angle_embedding.npy")

In [11]:
bob.shape, bobby.shape

((415, 26), (415, 26))

In [12]:
from dtw import dtw
from scipy.spatial.distance import cosine
dist, cost, acc, path = dtw(bob, bobby, dist=lambda x, y: cosine(x, y))  # Cosine similarity per frame
dist, cost, acc, path

(0.0,
 array([[0.        , 0.03771295, 0.09852478, ..., 1.25235135, 1.23231894,
         1.19858317],
        [0.03771295, 0.        , 0.01729563, ..., 1.13944192, 1.11458682,
         1.06613932],
        [0.09852478, 0.01729563, 0.        , ..., 1.05138301, 1.02602706,
         0.97050269],
        ...,
        [1.25235135, 1.13944192, 1.05138301, ..., 0.        , 0.00592921,
         0.02203968],
        [1.23231894, 1.11458682, 1.02602706, ..., 0.00592921, 0.        ,
         0.00879113],
        [1.19858317, 1.06613932, 0.97050269, ..., 0.02203968, 0.00879113,
         0.        ]]),
 array([[0.00000000e+00, 3.77129450e-02, 1.36237729e-01, ...,
         3.75760462e+02, 3.76992781e+02, 3.78191364e+02],
        [3.77129450e-02, 0.00000000e+00, 1.72956314e-02, ...,
         3.67110431e+02, 3.68225018e+02, 3.69291158e+02],
        [1.36237729e-01, 1.72956314e-02, 0.00000000e+00, ...,
         3.63374090e+02, 3.64400117e+02, 3.65370620e+02],
        ...,
        [3.75760462e+02, 3.671